# 🏛️ Conflict of Interest (COI) Matrix Generator for Academic Peer Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/visvesh-sathe/reviewer_assignment/blob/main/coi_generator/coi_generator.ipynb)

> ⚠️ **IMPORTANT - GPU RUNTIME REQUIRED:**
> To run the local LLM with high performance, please set your Colab runtime to GPU:
> **Runtime -> Change runtime type -> T4 GPU** (Hardware accelerator: T4 GPU).

This standalone Colab workflow automates the extraction of paper author affiliations and evaluates institutional Conflicts of Interest (COI) between reviewers and submitted papers using **local Ollama (`qwen2.5:7b`) with ZERO external API keys or logins**.

### 🦙 Why a Local LLM (`qwen2.5:7b`) via Ollama?
Institutional affiliations in conference submissions frequently use ambiguous acronyms and abbreviations (e.g., **UIUC** ↔ *University of Illinois Urbana-Champaign*, **MIT** ↔ *Massachusetts Institute of Technology*, **GT** ↔ *Georgia Tech*, **Intel Labs** ↔ *Intel*). Pure Python string matching often misses these conflicts. By running `qwen2.5:7b` locally in Ollama on a T4 GPU, you get full institutional reasoning with complete privacy and zero API setup.

---
### 📋 5-Step Workflow Overview:
1. **Setup:** Install Ollama, start daemon, and pull `qwen2.5:7b`
2. **Step 1:** Download `cfg_reviewer_expertise.csv` template
3. **Step 2:** Upload your edited `reviewer_expertise.csv`
4. **Step 3:** Upload your submitted `cfg_paper_info.csv`
5. **Step 4:** Extract & scrub affiliations into `paper_id_and_affiliations.csv` (Python pre-processing)
6. **Step 5:** Run `affiliation-matcher` skill locally against Ollama & download `cfg_coi.csv`

---
## ⚙️ Setup: Install Ollama & Pull `qwen2.5:7b` Local Model
Installs Ollama, starts the `ollama serve` daemon in the background, waits until it is ready, and downloads `qwen2.5:7b` onto the local T4 GPU.

In [ ]:
#@title Setup Ollama & Pull qwen2.5:7b (Zero API Keys Required) { display-mode: "form" }
import os
import sys
import time
import shutil
import subprocess
import urllib.request

# 1. Install prerequisites (zstd required by Ollama installer) and Ollama
print("📦 Installing zstd and Ollama...")
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Ensure /usr/local/bin is in PATH and resolve ollama binary
if "/usr/local/bin" not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"/usr/local/bin:{os.environ.get('PATH', '')}"
ollama_bin = shutil.which("ollama") or "/usr/local/bin/ollama"

if not os.path.exists(ollama_bin):
    raise RuntimeError(f"❌ Ollama binary not found at '{ollama_bin}' after installation.")

# 2. Start Ollama daemon in background
print("\n🚀 Starting Ollama background server...")
log_file = open("ollama_server.log", "w")
subprocess.Popen([ollama_bin, "serve"], stdout=log_file, stderr=log_file)

# 3. Wait for Ollama server to become ready
print("⏳ Waiting for Ollama server to become ready on http://localhost:11434...")
ready = False
for attempt in range(40):
    try:
        with urllib.request.urlopen("http://localhost:11434/api/version", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                print("✅ Ollama server is up and listening on http://localhost:11434")
                break
    except Exception:
        time.sleep(1)

if not ready:
    print("⚠️ Warning: Ollama did not respond within 40 seconds. Server log:")
    if os.path.exists("ollama_server.log"):
        with open("ollama_server.log", "r") as lf:
            print(lf.read()[-500:])

# 4. Pull qwen2.5:7b
print("\n📥 Pulling qwen2.5:7b model (optimized for T4 GPU)...\n")
!{ollama_bin} pull qwen2.5:7b

# 5. Initialize workspace directories
os.makedirs("input_files", exist_ok=True)
os.makedirs("intermediate_files", exist_ok=True)
os.makedirs("output_files", exist_ok=True)
os.makedirs(".agents/skills/affiliation-matcher", exist_ok=True)

# Fetch repository scripts if running in fresh Colab session
if not os.path.exists("coi_generation.py"):
    print("\n📡 Fetching latest coi_generator scripts and skills from GitHub...")
    !git clone --depth 1 https://github.com/visvesh-sathe/reviewer_assignment.git repo_tmp
    !cp -r repo_tmp/coi_generator/* . 2>/dev/null || true
    !cp -r repo_tmp/.agents . 2>/dev/null || true
    !rm -rf repo_tmp

print("\n🎉 Ollama and workspace setup complete! Ready for Step 1.")


---
## 📥 Step 1: Download Reviewer Expertise Template
Download the pre-structured template CSV to view or modify reviewer names, affiliations, and expertise ratings.

In [ ]:
#@title Step 1: Download cfg_reviewer_expertise.csv Template { display-mode: "form" }
import os

template_path = "input_files/cfg_reviewer_expertise.csv"

if os.path.exists(template_path):
    try:
        from google.colab import files
        print(f"⬇️ Triggering download for {template_path}...")
        files.download(template_path)
    except ImportError:
        print(f"✅ Running locally: Template is ready at {template_path}")
else:
    print(f"⚠️ Template file {template_path} not found. Please run the setup cell above.")

---
## 📤 Step 2: Upload Edited Reviewer Expertise
Upload your edited `reviewer_expertise.csv` (or `cfg_reviewer_expertise.csv`). The file will be saved to `input_files/cfg_reviewer_expertise.csv`.

In [ ]:
#@title Step 2: Upload Reviewer Expertise CSV { display-mode: "form" }
import os
import shutil
import pandas as pd

target_exp_path = "input_files/cfg_reviewer_expertise.csv"

try:
    from google.colab import files
    print("📤 Please upload your reviewer expertise CSV file (e.g., reviewer_expertise.csv):")
    uploaded = files.upload()
    for fname in uploaded.keys():
        shutil.move(fname, target_exp_path)
        print(f"✅ Saved uploaded file as '{target_exp_path}'")
        break
except ImportError:
    print(f"✅ Running locally: Using existing {target_exp_path}")

if os.path.exists(target_exp_path):
    df_exp = pd.read_csv(target_exp_path)
    print(f"\n📊 Loaded {len(df_exp)} reviewers. Preview:")
    cols_to_show = [c for c in df_exp.columns if c.lower() in ["firstname", "lastname", "affiliation", "first name", "last name"]]
    display(df_exp[cols_to_show].head(10))
else:
    print(f"❌ Error: {target_exp_path} not found.")

---
## 📤 Step 3: Upload Submitted Paper Information
Upload your submission metadata CSV (`cfg_paper_info.csv` or `paper_info.csv`) containing submitted paper IDs and author affiliations.

In [ ]:
#@title Step 3: Upload cfg_paper_info.csv { display-mode: "form" }
import os
import shutil
import pandas as pd

target_info_path = "input_files/cfg_paper_info.csv"

try:
    from google.colab import files
    print("📤 Please upload your paper info CSV file (e.g., cfg_paper_info.csv):")
    uploaded = files.upload()
    for fname in uploaded.keys():
        shutil.move(fname, target_info_path)
        print(f"✅ Saved uploaded file as '{target_info_path}'")
        break
except ImportError:
    print(f"✅ Running locally: Using existing {target_info_path}")

if os.path.exists(target_info_path):
    try:
        df_info = pd.read_csv(target_info_path)
    except UnicodeDecodeError:
        df_info = pd.read_csv(target_info_path, encoding="iso-8859-1")
    print(f"\n📊 Loaded paper info file with {len(df_info)} submissions across {len(df_info.columns)} columns.")
else:
    print(f"❌ Error: {target_info_path} not found.")

---
## 🧹 Step 4: Extract & Scrub Paper Affiliations (Python)
Runs the automated Python scrubber to extract paper IDs and unique institutional affiliations into `intermediate_files/paper_id_and_affiliations.csv`. All author names and paper titles are removed to preserve double-blind reviewing integrity.

In [ ]:
#@title Step 4: Extract & Scrub Paper Affiliations { display-mode: "form" }
import os
import pandas as pd
from coi_generation import scrub_affiliations

paper_info_path = "input_files/cfg_paper_info.csv"
scrub_output_path = "intermediate_files/paper_id_and_affiliations.csv"

if not os.path.exists(paper_info_path):
    raise FileNotFoundError(f"{paper_info_path} not found. Please upload it in Step 3.")

print(f"🔄 Scrubbing affiliations from '{paper_info_path}'...")
df_scrubbed = scrub_affiliations(paper_info_path, scrub_output_path)

print(f"\n✅ Double-blind scrubbed affiliations saved to '{scrub_output_path}'!")
print(f"📊 Preview of scrubbed affiliations (first 10 papers):")
display(df_scrubbed.head(10))

---
## 🤖 Step 5: Run `affiliation-matcher` Skill & Download Deliverables
Invokes the **`affiliation-matcher`** skill on the pre-processed output file (`intermediate_files/paper_id_and_affiliations.csv`). The skill runs headlessly against the local Ollama server (`http://localhost:11434/v1`) using `qwen2.5:7b` with auto-approved tool permissions.

In [ ]:
#@title Step 5: Execute affiliation-matcher Skill (Ollama qwen2.5:7b) { display-mode: "form" }
import os
import shutil
import pandas as pd

# Paths
scrub_path = "intermediate_files/paper_id_and_affiliations.csv"
reviewer_exp_path = "input_files/cfg_reviewer_expertise.csv"
output_coi_path = "output_files/cfg_coi.csv"
editable_coi_path = "input_files/cfg_coi.csv"
report_path = "output_files/coi_report.txt"

if not os.path.exists(scrub_path):
    raise FileNotFoundError(f"Missing pre-processed file: {scrub_path}. Please run Step 4 first.")
if not os.path.exists(reviewer_exp_path):
    raise FileNotFoundError(f"Missing reviewer file: {reviewer_exp_path}. Please complete Step 2.")

print("🤖 Running 'affiliation-matcher' skill headlessly against local Ollama (qwen2.5:7b)...")
print("   • Endpoint:     http://localhost:11434/v1")
print("   • Model:        qwen2.5:7b")
print("   • Pre-processed input: intermediate_files/paper_id_and_affiliations.csv")
print("   • Reviewers:    input_files/cfg_reviewer_expertise.csv\n")

# Execute skill headlessly against local Ollama server with auto-approved permissions
!OPENAI_BASE_URL="http://localhost:11434/v1" OPENAI_API_KEY="ollama" agy -p "/affiliation-matcher" --model qwen2.5:7b --dangerously-skip-permissions

# Fallback / verification helper: Ensure output_files/cfg_coi.csv and coi_report.txt exist
if not os.path.exists(output_coi_path) or os.path.getsize(output_coi_path) == 0:
    print("\n🔄 Running affiliation-matcher helper via local Ollama endpoint...")
    from coi_generation import evaluate_coi_with_ollama, build_coi_matrix_and_report
    conflicts = evaluate_coi_with_ollama(
        paper_affiliations_path=scrub_path,
        reviewer_exp_path=reviewer_exp_path,
        base_url="http://localhost:11434/v1",
        model="qwen2.5:7b"
    )
    coi_df = build_coi_matrix_and_report(
        conflicts=conflicts,
        paper_affiliations_path=scrub_path,
        reviewer_exp_path=reviewer_exp_path,
        output_coi_path=output_coi_path,
        report_path=report_path,
        editable_coi_path=editable_coi_path
    )
else:
    coi_df = pd.read_csv(output_coi_path, index_col=0)

print(f"\n✅ COI Matrix successfully generated with {(coi_df == 1).sum().sum()} total conflicts detected.")

# Preview Audit Report
if os.path.exists(report_path):
    print("\n" + "="*50)
    print("📄 COI AUDIT REPORT PREVIEW:")
    print("="*50)
    with open(report_path, "r") as f:
        lines = f.readlines()
        print("".join(lines[:35]))
        if len(lines) > 35:
            print(f"... ({len(lines) - 35} more lines in {report_path})")

# Trigger automatic file downloads
try:
    from google.colab import files
    print(f"\n⬇️ Downloading {output_coi_path} to your computer...")
    files.download(output_coi_path)
    if os.path.exists(report_path):
        print(f"⬇️ Downloading {report_path}...")
        files.download(report_path)
except ImportError:
    print(f"\n✅ Running locally: Deliverables saved to {output_coi_path} and {report_path}")
